# Lab 3.8 &mdash; Challenge &mdash; The Leave Request Workflow

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Assemble everything: state, routing, a cycle, a budget, checkpoints and a gate
- Gate <i>one branch</i> rather than the whole graph &mdash; routing, not a bigger interrupt
- Run four requests down four different paths through one compiled graph

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-08")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## The brief

One graph that handles every request in the case file.

```
START -> validate --(incomplete, budget left)--> ask ---+
            |                                            |   (cycle)
            |  <-----------------------------------------+
            |--(incomplete, budget spent)--> withdraw -> END
            |
            +--(complete)--> assess --(short, covered)------> notify -> END
                                |
                                +--(long or uncovered)--> manager_review -> notify -> END
                                                          ^ paused here
```

Four requirements:

1. **A cycle with a budget.** `LV-5002` has no reason. Ask, and give up after
   `POLICY["max_clarifications"]`.
2. **A checkpointer**, so a paused request can be picked up later.
3. **A gate on one branch only.** `manager_review` pauses; the auto-approval branch must not.
   You do this with *routing*, not with a bigger `interrupt_before`.
4. **`notify` is irreversible** and must never run before its branch's approval.

## Section 1 &mdash; The two routers

`validate` decides whether we can proceed at all. `assess` decides who has to say yes.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

SENT = []


class LeaveState(TypedDict):
    request_id: str
    reason: str
    complete: bool
    attempts: int
    inbox: dict
    verdict: str
    approved_by: str
    notes: Annotated[list, add]


def after_validate(state: LeaveState) -> str:
    if state["complete"]:
        return "assess"
    if state["attempts"] >= POLICY["max_clarifications"]:
        return BLANK        # TODO: we have asked enough times -- which branch?
    return BLANK            # TODO: still incomplete, budget left -- which branch?


def after_assess(state: LeaveState) -> str:
    """Short and covered goes straight out. Anything else needs a person."""
    r = REQUESTS[state["request_id"]]
    if r["days"] <= POLICY["manager_over_days"] and r["balance"] >= r["days"]:
        return "auto"
    return BLANK            # TODO: which branch ends up in front of the manager?

In [ ]:
# --- Self-check: Section 1   (two pure functions)
check("validate routes: complete -> assess, budget left -> ask, budget spent -> withdraw",
      lambda: (after_validate({"complete": True, "attempts": 0}),
               after_validate({"complete": False, "attempts": 0}),
               after_validate({"complete": False, "attempts": 2})) == ("assess", "ask", "withdraw"))
check("assess routes LV-5005 to auto, LV-5004 and LV-5003 to review",
      lambda: (after_assess({"request_id": "LV-5005"}),
               after_assess({"request_id": "LV-5004"}),
               after_assess({"request_id": "LV-5003"})) == ("auto", "review", "review"),
      "LV-5003 is short but has no balance, so a person decides")
score()

## Section 2 &mdash; The graph

Nodes are given &mdash; you have written all of them before. Wire the cycle, and put the gate on the
branch that needs it.

In [ ]:
def validate(state):
    reason = state["reason"] or REQUESTS[state["request_id"]]["reason"]
    return {"reason": reason, "complete": bool(reason.strip()),
            "notes": [f'validate(attempt={state["attempts"]})']}

def ask_employee(state):
    n = state["attempts"] + 1
    return {"attempts": n, "reason": state["inbox"].get(n, ""), "notes": [f"ask({n})"]}

def withdraw(state):
    return {"verdict": "withdrawn", "notes": ["withdraw"]}

def assess(state):
    return {"notes": ["assess"]}

def auto(state):
    return {"verdict": "granted", "approved_by": "policy", "notes": ["auto"]}

def manager_review(state):
    """Paused before this node. The approver's edit lands in the checkpoint."""
    return {"verdict": state["verdict"] or "granted",
            "approved_by": state["approved_by"] or REQUESTS[state["request_id"]]["manager"],
            "notes": ["manager_review"]}

def notify(state):
    """Irreversible."""
    SENT.append(f'{REQUESTS[state["request_id"]]["who"]}: {state["verdict"]} '
                f'by {state["approved_by"] or "NOBODY"}')
    return {"notes": [f"notify -> {SENT[-1]}"]}


def build_workflow():
    b = StateGraph(LeaveState)
    for name, fn in [("validate", validate), ("ask", ask_employee), ("withdraw", withdraw),
                     ("assess", assess), ("auto", auto),
                     ("manager_review", manager_review), ("notify", notify)]:
        b.add_node(name, fn)

    b.add_edge(START, "validate")
    b.add_conditional_edges("validate", after_validate,
                            {"ask": "ask", "withdraw": "withdraw", "assess": "assess"})
    b.add_edge("ask", BLANK)              # TODO: the cycle

    b.add_conditional_edges("assess", after_assess,
                            {"auto": "auto", "review": "manager_review"})
    b.add_edge("auto", "notify")
    b.add_edge("manager_review", "notify")
    b.add_edge("notify", END)
    b.add_edge("withdraw", END)

    return b.compile(checkpointer=InMemorySaver(),
                     interrupt_before=BLANK)   # TODO: gate ONE branch, not the whole graph

In [ ]:
# --- Self-check: Section 2   (four requests, four paths, one graph -- no model)
BASE = {"reason": "", "complete": False, "attempts": 0, "inbox": {},
        "verdict": "", "approved_by": "", "notes": []}

def start(rid, inbox=None):
    SENT.clear()
    app, cfg = build_workflow(), {"configurable": {"thread_id": rid}}
    app.invoke({**BASE, "request_id": rid, "inbox": inbox or {}}, cfg)
    return app, cfg

check("LV-5005 is short and covered: it runs straight through, no pause",
      lambda: (start("LV-5005")[0].get_state(start("LV-5005")[1]).next == ()
               and len(SENT) == 1))
check("LV-5004 is long: it pauses before manager_review and sends nothing",
      lambda: (start("LV-5004")[0].get_state(start("LV-5004")[1]).next == ("manager_review",)
               and SENT == []),
      "gate the branch, not the graph -- LV-5005 must not have paused")
check("LV-5002 has no reason and never replies: it withdraws inside the budget",
      lambda: start("LV-5002")[0].get_state(start("LV-5002")[1]).values["verdict"] == "withdrawn")
check("LV-5002 replying on attempt 2 gets through validation and reaches the gate",
      lambda: start("LV-5002", {2: "conference"})[0]
              .get_state(start("LV-5002", {2: "conference"})[1]).next == ("manager_review",))
score()

## Section 3 &mdash; Approve one, and resume

In [ ]:
def approve(app, cfg, who: str, verdict: str):
    app.update_state(cfg, {"verdict": verdict, "approved_by": who,
                           "notes": [f"{who} decided {verdict}"]})
    return app.invoke(BLANK, cfg)     # TODO: continue the paused run

In [ ]:
# --- Self-check: Section 3
def approved(verdict="granted"):
    app, cfg = start("LV-5004")
    approve(app, cfg, "Sam O.", verdict)
    return app.get_state(cfg).values

check("approving resumes the run and notify finally fires",
      lambda: (approved(), len(SENT) == 1)[1])
check("a refusal is what goes out, and the approver is named",
      lambda: (approved("refused"), "refused" in SENT[-1] and "Sam O." in SENT[-1])[1],
      "the person overrides the graph, and the record says who")
check("the trail contains the approver's own line",
      lambda: any("Sam O. decided" in n for n in approved()["notes"]))
score()

## Watch it run

Every request in the case file through one compiled graph.

In [ ]:
def walk_every_request():
    for rid in sorted(REQUESTS):
        app, cfg = start(rid)
        s = app.get_state(cfg)
        if s.next == ("manager_review",):
            approve(app, cfg, REQUESTS[rid]["manager"], "granted")
            s = app.get_state(cfg)
            tag = "paused, then approved"
        else:
            tag = "no pause needed"
        print(f"{rid}  {tag:22} {' -> '.join(s.values['notes'][-6:])}")
        print(f"{'':10}verdict={s.values['verdict']!r} sent={SENT}")

guard(walk_every_request)

## Run it for real &mdash; the model writes what the employee reads

In [ ]:
def live_workflow():
    app, cfg = start("LV-5004")
    r = REQUESTS["LV-5004"]

    print("brief for the manager:", ask(
        f'One sentence for a manager deciding on leave: {r["who"]} ({r["days"]} days '
        f'{r["kind"]}, reason "{r["reason"]}", balance {r["balance"]}).',
        system="You brief a busy manager. One sentence, no greeting.").strip()[:220])

    approve(app, cfg, r["manager"], "granted")
    print("\nmessage to the employee:", ask(
        f'Tell {r["who"]} their {r["days"]}-day leave was granted by {r["manager"]}. '
        f'One short sentence, no greeting.',
        system="You write brief internal HR messages.").strip()[:220])

    print("\nthe record:")
    for n in app.get_state(cfg).values["notes"]:
        print("   ", n)

if llm_ready():
    guard(live_workflow)

### Read it

**The gate is on a branch, not on the graph.** `interrupt_before=["manager_review"]` pauses only
requests routed there. `LV-5005` never stopped. If you had gated `notify` instead, every
auto-approval would sit waiting for a manager who has nothing to decide &mdash; the fastest way to
make people ignore an approval queue.

**Four behaviours, one state object.** The cycle, the budget, the conditional gate and the audit
trail are all fields in `LeaveState` plus edges. Nothing here needed a framework feature you have
not already used in this module.

**What you take from Module 3:** state you can print and assert on; nodes and edges as ordinary
testable code; conditional routing; reducers for the fields that accumulate; a cycle with a budget;
checkpoints that make resume, approval, rewind and audit possible. Module 5 puts several agents
into exactly this shape, and its supervisor is `after_assess` with a model choosing the string.

In [ ]:
score()

## Your turn

1. `POLICY["hr_over_days"]` is 10 and nothing uses it. Add an `hr_review` node so `LV-5004` needs
   two approvals, in order. You will need a second gate and a second resume.
2. Rewind an approved request to before `manager_review` and run it forward. It pauses again &mdash;
   see Lab 3.7. Make the approval survive the rewind by having the gate read a fact in the state.
3. Swap `InMemorySaver` for `SqliteSaver`, run `LV-5004`, restart the kernel, and resume the paused
   request from a fresh process. That is the whole production migration.